In [1]:
import pandas as pd
import duckdb
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn import set_config

set_config(display="text")

## Predicting Passenger Destinations (Behavioral AI)
**Objective:** Predicting the final drop-off borough of a passenger based only on where and when they are picked up, allowing fleet managers to anticipate vehicle flow and strategically position assets.

- Drivers picked up in residential zones face the risk of "deadheading"—driving to a remote location where they cannot find a return passenger.

- Anticipating a driver's destination allows dispatchers to "chain" rides, ensuring a driver heading to the Financial District is immediately queued for a high-yield corporate trip upon arrival.

## Hypotheses:

1. The Commuter Pipeline: Human commuting behavior exhibits highly predictable spatial-temporal patterns. By feeding an algorithm the pickup location and time of day, it can successfully classify the intended destination without knowing the physical distance.

2. The Manhattan Imbalance: Because NYC taxi data is heavily skewed toward intra-Manhattan travel, classification models face severe "Class Imbalance." The model will naturally excel at predicting Manhattan drop-offs but will require advanced handling to accurately predict outer-borough travel.

In [3]:
# 1. Load the pristine data
con = duckdb.connect()
clean_data_path = 'data/cleaned_taxi_data_for_ml.parquet'
df = con.execute(f"SELECT * FROM '{clean_data_path}'").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [5]:
# 2. SELECT FEATURES: Where are they, what time is it, and what day is it?
# Notice we DO NOT use distance, because the driver doesn't know the distance until the destination is declared!
features = ['pickup_hour', 'day_of_week', 'pickup_borough']
X_raw = df[features]
y = df['dropoff_borough'] # The Category we want to predict

In [7]:
# 3. ONE-HOT ENCODING
print("🧮 Encoding locations...")
# We keep drop_first=False here so the model explicitly sees every starting borough
X = pd.get_dummies(X_raw, columns=['pickup_borough'], drop_first=False)

🧮 Encoding locations...


In [9]:
# Sample 150,000 trips
X_sample = X.sample(n=150000, random_state=42)
y_sample = y.loc[X_sample.index]
X_train, X_test, y_train, y_test = train_test_split(X_sample, y_sample, test_size=0.2, random_state=42)

In [11]:
# 4. TRAIN THE CLASSIFIER
print("🧠 Training the Random Forest Classifier... (Please wait)")
rf_classifier = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf_classifier.fit(X_train, y_train)

🧠 Training the Random Forest Classifier... (Please wait)


RandomForestClassifier(max_depth=10, n_jobs=-1, random_state=42)

In [13]:
# 5. GRADE THE EXAM
preds = rf_classifier.predict(X_test)
acc = accuracy_score(y_test, preds)

In [15]:
print("\n🎯 --- BEHAVIORAL AI REPORT CARD --- 🎯")
print(f"Overall Accuracy: {acc * 100:.2f}%\n")

# A Classification Report breaks down the accuracy for EACH specific borough
print("Detailed Classification Report:")
print(classification_report(y_test, preds, zero_division=0))


🎯 --- BEHAVIORAL AI REPORT CARD --- 🎯
Overall Accuracy: 88.78%

Detailed Classification Report:
               precision    recall  f1-score   support

        Bronx       1.00      0.00      0.01       326
     Brooklyn       0.56      0.20      0.30      1535
          EWR       0.00      0.00      0.00        66
    Manhattan       0.90      0.99      0.94     26436
          N/A       0.00      0.00      0.00       125
       Queens       0.43      0.05      0.09      1507
Staten Island       0.00      0.00      0.00         5

     accuracy                           0.89     30000
    macro avg       0.41      0.18      0.19     30000
 weighted avg       0.85      0.89      0.85     30000



In [18]:
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

print("⚖️ Balancing the data using SMOTE...")

# 1. Apply SMOTE only to the TRAINING data (Never the test data!)
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

print(f"Original training rows: {len(X_train):,}")
print(f"SMOTE balanced training rows: {len(X_train_balanced):,}")

# 2. TRAIN THE CLASSIFIER on the new balanced data
print("🧠 Training the Random Forest Classifier... (This will take a bit longer now!)")
rf_classifier = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf_classifier.fit(X_train_balanced, y_train_balanced)

# 3. GRADE THE EXAM on the original, untouched Test data
preds = rf_classifier.predict(X_test)

print("\n🎯 --- BALANCED BEHAVIORAL AI REPORT CARD --- 🎯")
# Overall accuracy will drop, but watch the outer-borough recall scores!
print(classification_report(y_test, preds, zero_division=0))

⚖️ Balancing the data using SMOTE...


C:\Users\Ahmad Reza\anaconda3\Lib\site-packages\sklearn\base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


Original training rows: 120,000
SMOTE balanced training rows: 739,375
🧠 Training the Random Forest Classifier... (This will take a bit longer now!)

🎯 --- BALANCED BEHAVIORAL AI REPORT CARD --- 🎯
               precision    recall  f1-score   support

        Bronx       0.02      0.06      0.03       326
     Brooklyn       0.19      0.24      0.21      1535
          EWR       0.00      0.47      0.01        66
    Manhattan       0.94      0.42      0.58     26436
          N/A       0.03      0.52      0.05       125
       Queens       0.21      0.11      0.14      1507
Staten Island       0.00      0.20      0.00         5

     accuracy                           0.39     30000
    macro avg       0.20      0.29      0.15     30000
 weighted avg       0.85      0.39      0.53     30000

